# Step 6, Digitalization as a Cross-Sectoral Theme

**Goal:** Identify digitalization-related laws in the SGBS and test whether they constitute a cross-sectoral
*Querschnittsthema* or are concentrated in specific Hauptgruppen.

**Research sub-question:** *Does the SGBS contain a latent 'digitalization' theme that cuts across the official administrative classification?*

Method: keyword-based scoring on lemmatized text. A law is 'digitalization-related' if it contains
tokens from >= 2 distinct keyword categories. Results are cross-tabulated with the 9 official
Hauptgruppen and visualized in t-SNE space.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

CLEAN_DATA_PATH    = "../data/processed/sgbs_clean.csv"
TSNE_TFIDF_PATH    = "../data/processed/tsne_tfidf_2d.npy"
TSNE_W2V_PATH      = "../data/processed/tsne_w2v_2d.npy"
KM_TFIDF_9_PATH    = "../data/processed/kmeans_labels_tfidf_k9.npy"

os.makedirs("../report/figures", exist_ok=True)

df = pd.read_csv(CLEAN_DATA_PATH)

HAUPTGRUPPEN = {
    '1': 'Staatsrecht/Org.',
    '2': 'Zivilrecht/Strafrecht',
    '3': 'Gesundheit',
    '4': 'Erziehung/Wiss.',
    '5': 'Polizei/Militaer',
    '6': 'Finanzen/Lieg.',
    '7': 'Bau/Energie/Umwelt',
    '8': 'Arbeit/Soziales',
    '9': 'Wirtschaft/Verkehr',
}
df['hauptgruppe'] = df['systematic_number'].astype(str).str[0].apply(
    lambda x: x if x in HAUPTGRUPPEN else 'S'
)

X_tsne     = np.load(TSNE_TFIDF_PATH)
X_tsne_w2v = np.load(TSNE_W2V_PATH)
km_labels  = np.load(KM_TFIDF_9_PATH)

print(f"Corpus: {len(df)} documents")

## 2. Keyword Scoring

I define eight keyword categories covering distinct aspects of digitalization.
Matching is case-insensitive substring search on the lemmatized `text_clean` column,
which catches compound nouns (e.g. 'Datenschutzgesetz' matches 'datenschutz').
A law is flagged as digitalization-related if it matches >= 2 categories.

In [ ]:
DIGI_KEYWORDS = {
    'digital':      'digital',
    'elektronisch': 'elektronisch',
    'Datenschutz':  'datenschutz',
    'Informatik':   'informatik',
    'Internet':     'internet',
    'online':       'online',
    'Plattform':    'plattform',
    'Software':     'software',
}

text_col = df['text_clean'].fillna('').str.lower()

score_df = pd.DataFrame({
    cat: text_col.str.contains(pat, na=False).astype(int)
    for cat, pat in DIGI_KEYWORDS.items()
})

df['digi_score'] = score_df.sum(axis=1)
df['is_digi']    = df['digi_score'] >= 2

print(f"Digitalization-related (score >= 2): {df['is_digi'].sum()} / {len(df)} ({df['is_digi'].mean()*100:.1f}%)")
print()
print(f"{'Keyword':15s}  {'Docs':>5}  {'%':>6}")
print("-" * 30)
for cat in DIGI_KEYWORDS:
    h = int(score_df[cat].sum())
    print(f"{cat:15s}  {h:5d}  {h/len(df)*100:5.1f}%")

## 3. Distribution by Hauptgruppe

In [ ]:
hgs      = sorted(HAUPTGRUPPEN.keys())
hg_total = df.groupby('hauptgruppe').size()
hg_digi  = df[df['is_digi']].groupby('hauptgruppe').size()

hg_stats = pd.DataFrame({
    'label': [HAUPTGRUPPEN[k] for k in hgs],
    'total': [int(hg_total.get(k, 0)) for k in hgs],
    'digi':  [int(hg_digi.get(k, 0))  for k in hgs],
}, index=hgs)
hg_stats['pct'] = hg_stats['digi'] / hg_stats['total'] * 100
hg_stats = hg_stats.sort_values('pct', ascending=False)

avg_pct = df['is_digi'].mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hg_abs = hg_stats.sort_values('digi', ascending=True)
axes[0].barh(hg_abs['label'], hg_abs['digi'],
             color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Number of digitalization-related laws')
axes[0].set_title('Digitalization laws per Hauptgruppe (count)')
for i, (_, row) in enumerate(hg_abs.iterrows()):
    axes[0].text(row['digi'] + 0.3, i, str(int(row['digi'])), va='center', fontsize=8)

hg_pct = hg_stats.sort_values('pct', ascending=True)
axes[1].barh(hg_pct['label'], hg_pct['pct'],
             color='steelblue', alpha=0.85, edgecolor='white')
axes[1].axvline(avg_pct, color='firebrick', linestyle='--', alpha=0.7,
                label=f'Corpus avg {avg_pct:.1f}%')
axes[1].set_xlabel('Share of Hauptgruppe that is digitalization-related (%)')
axes[1].set_title('Digitalization penetration by Hauptgruppe (%)')
axes[1].legend(fontsize=8)
for i, (_, row) in enumerate(hg_pct.iterrows()):
    axes[1].text(row['pct'] + 0.3, i, f"{row['pct']:.1f}%", va='center', fontsize=8)

plt.suptitle('Digitalization in the SGBS -- Distribution across Hauptgruppen', fontsize=12)
plt.tight_layout()
plt.savefig('../report/figures/06_digitalization_hg.png', dpi=150)
plt.show()
print(f"Digitalization laws: {df['is_digi'].sum()} / {len(df)} ({avg_pct:.1f}%)")
print(f"Present in all 9 HGs: {(hg_stats['digi'] > 0).all()}")

## 4. t-SNE Visualization, Are Digitalization Laws Spatially Clustered?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_data, title in [
    (axes[0], X_tsne,     'TF-IDF'),
    (axes[1], X_w2v_tsne, 'Word2Vec'),
]:
    colors = ['steelblue' if d else '#cccccc' for d in df['is_digi']]
    ax.scatter(X_data[:, 0], X_data[:, 1], c=colors, s=8, alpha=0.7)
    ax.set_title(f't-SNE ({title}) — digitalization-related laws highlighted')
    ax.set_xticks([]); ax.set_yticks([])

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label=f'Digitalization-related (n={df["is_digi"].sum()})'),
    Patch(facecolor='#cccccc',   label=f'Other (n={(~df["is_digi"]).sum()})')
]
axes[0].legend(handles=legend_elements, fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('../report/figures/06_tsne_digitalization.png', dpi=150)
plt.show()
print('Figure saved: 06_tsne_digitalization.png')

## 5. Distribution across K-Means Clusters (TF-IDF k=9)

In [ ]:
df['km_cluster'] = km_labels

cluster_total = df.groupby('km_cluster').size()
cluster_digi  = (df[df['is_digi']].groupby('km_cluster').size()
                  .reindex(cluster_total.index, fill_value=0))

km_stats = pd.DataFrame({
    'total': cluster_total,
    'digi':  cluster_digi,
})
km_stats['pct'] = km_stats['digi'] / km_stats['total'] * 100

print(f"{'Cluster':>8}  {'Digi':>5}  {'Total':>6}  {'%':>6}")
print('-' * 32)
for c, row in km_stats.iterrows():
    print(f"C{c:>7}  {row['digi']:5.0f}  {row['total']:6.0f}  {row['pct']:5.1f}%")

spread_entropy = -(km_stats['digi']/km_stats['digi'].sum() *
                   (km_stats['digi']/km_stats['digi'].sum()).apply(
                       lambda p: 0 if p == 0 else __import__('math').log2(p))).sum()
max_entropy = __import__('math').log2(len(km_stats))
print(f"\nSpread-Entropy: {spread_entropy:.2f} / {max_entropy:.2f} max = {spread_entropy/max_entropy*100:.0f}%")